# 07 · Aşama 8 — Sağlamlık: gürültü ve eksik öznitelikler
Test girdileri bozulur (etiketler asla değiştirilmez). İki eşik politikası karşılaştırılır:
**devralınan eşik** (temiz kalibrasyondan) ve **yeniden kalibrasyon** (aynı bozulmaya uğramış kalibrasyon verisiyle).
* Gauss gürültüsü σ ∈ {0,01; 0,05; 0,10} (ölçeklenmiş öznitelik uzayında).
* Eksik öznitelikler %5 / %10 / %20 → normal-eğitim medyanıyla doldurma.
Model: merkezi AE, tohum 0 (kayıtlı model). 3 rastgele tekrar.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
import tensorflow as tf
from trustids import data as D, models as M, conformal as C
P = D.load_prepared(ART / "prepared"); X = P["X"]
m = tf.keras.models.load_model(ART / "models" / "ae_seed0.keras")
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_phase8_robustness")
MED = np.median(X["benign_train"], axis=0)
th0 = C.thresholds(M.ae_score(m, X["benign_cal"]), cfg.alpha, cfg.beta)
def corrupt(Xs, kind, level, rng):
    if kind == "gürültü": return Xs + rng.normal(0, level, Xs.shape)
    Z = Xs.copy(); mask = rng.random(Xs.shape) < level; Z[mask] = np.broadcast_to(MED, Xs.shape)[mask]; return Z
rows = []
for kind, levels in [("temiz", [0.0]), ("gürültü", [0.01, 0.05, 0.10]), ("eksik", [0.05, 0.10, 0.20])]:
    for lv in levels:
        for rep in range(1 if kind == "temiz" else 3):
            rng = np.random.default_rng(100 * rep + int(lv * 1000))
            Z = {sp: (X[sp] if kind == "temiz" else corrupt(X[sp], kind, lv, rng)) for sp in ["benign_cal", "benign_test", "heldout_test", "dev_test"]}
            S = {sp: M.ae_score(m, Z[sp]) for sp in Z}
            for pol, th in [("devralınan eşik", th0), ("yeniden kalibrasyon", C.thresholds(S["benign_cal"], cfg.alpha, cfg.beta))]:
                rows.append({"bozulma": kind, "düzey": lv, "tekrar": rep, "politika": pol, "normal_FPR": np.mean(S["benign_test"] > th["tau_H"]),
                             "görülmemiş_ANOMALY": np.mean(S["heldout_test"] > th["tau_H"]),
                             "görülmemiş_ANOMALY+UNCERTAIN": np.mean(S["heldout_test"] > th["tau_L"]), "bilinen_ANOMALY": np.mean(S["dev_test"] > th["tau_H"])})
ROB = pd.DataFrame(rows); ROB.to_csv(RUN / "07_robustness_all.csv", index=False)
RS = ROB.groupby(["bozulma", "düzey", "politika"])[["normal_FPR", "görülmemiş_ANOMALY", "görülmemiş_ANOMALY+UNCERTAIN", "bilinen_ANOMALY"]].mean().round(4)
RS.to_csv(RUN / "07_robustness_summary.csv"); RS

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, kind in zip(axs, ["gürültü", "eksik"]):
    for pol, mk in [("devralınan eşik", "o-"), ("yeniden kalibrasyon", "s--")]:
        g = ROB[(ROB["bozulma"] == kind) & (ROB["politika"] == pol)].groupby("düzey")["normal_FPR"].mean() * 100
        ax.plot(g.index, g.values, mk, label=pol)
    ax.axhline(cfg.alpha * 100, color="k", ls=":", label="α"); ax.set_title(kind); ax.set_xlabel("düzey")
axs[0].set_ylabel("normal trafikte yanlış alarm (%)"); axs[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG / "07_robustness_fpr.png", dpi=150)
cfg.save(RUN / "config_used.json"); print("sonuçlar:", RUN)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")